# LeetCode 222 — Count Complete Tree Nodes bằng Python

Link đề: https://leetcode.com/problems/count-complete-tree-nodes/ · Độ khó: **Medium** · Chủ đề: Binary Tree, Binary Search, Bit Manipulation, Divide and Conquer

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def countNodes(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code (cây dựng theo số nút `n` biết trước nên so được với đáp án đúng `n`, và với một lời giải duyệt cây độc lập).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm **số nút** của một **cây nhị phân hoàn chỉnh (complete binary tree)** — và đề yêu cầu thuật toán chạy **dưới `O(n)`**.
2. Phân biệt **cây hoàn chỉnh**, **cây đầy đủ/hoàn hảo (perfect)** và **cây nhị phân bất kỳ**; nhớ công thức **cây hoàn hảo `h` tầng có `2^h - 1` nút**.
3. Đi từ cách **duyệt mọi nút `O(n)`** (DFS, BFS) → **dùng chiều cao hai gai trái/phải** để nhận ra cây con hoàn hảo, `O(log² n)` → **chặt nhị phân trên hàng lá cuối**, `O(log² n)` → bản **lặp không đệ quy**, `O(1)` bộ nhớ.
4. Tự dựng `TreeNode`, hàm **`build_tree` từ danh sách level-order** (theo định dạng LeetCode, `None` là nút rỗng) và hàm sinh **cây hoàn chỉnh theo số nút `n`** để test.
5. Tránh các lỗi kinh điển: áp dụng cho cây **không hoàn chỉnh**, nhầm `2^h` với `2^h - 1`, tính chiều cao sai một đơn vị, chặt nhị phân thiếu làm tròn lên (lặp vô hạn), quên `root is None`.

## 2. Đề bài & ví dụ

Cho `root` của một **cây nhị phân hoàn chỉnh (complete binary tree)**, trả về **số nút** của cây.

Theo định nghĩa: trong cây hoàn chỉnh, **mọi tầng đều được lấp đầy** trừ có thể **tầng cuối cùng**, và các nút ở tầng cuối **dồn về bên trái nhất có thể**. Tầng cuối `h` (đếm tầng gốc là `0`) có từ `1` đến `2^h` nút.

Hãy thiết kế thuật toán chạy trong thời gian **dưới `O(n)`**.

**Ví dụ 1**

- Input: `root = [1,2,3,4,5,6]`
- Output: `6`

**Ví dụ 2**

- Input: `root = []`
- Output: `0`

**Ví dụ 3**

- Input: `root = [1]`
- Output: `1`

**Ràng buộc:** số nút trong cây thuộc đoạn `[0, 5 * 10^4]`; `0 <= Node.val <= 5 * 10^4`; cây **được đảm bảo là hoàn chỉnh**.

### Lưu ý về cây trong notebook

LeetCode cung cấp sẵn lớp `TreeNode` và biểu diễn cây bằng danh sách theo **thứ tự từng tầng (level-order)**, dùng `null` cho nút rỗng. Trong notebook ta tự dựng lại: lớp `TreeNode`, hàm `build_tree(values)`, và hàm phụ để in cây. Bạn cần **đọc được** định dạng `[1,2,3,4,5,6]` thành cây:

```
        1
      /   \
     2     3
    / \   /
   4   5 6
```

In [1]:
from typing import Optional, List
from collections import deque
import random
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách level-order kiểu LeetCode (None = nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:      # con trái
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:      # con phải
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Chuyển cây về danh sách level-order (bỏ các None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([1, 2, 3, 4, 5, 6], 6),
    ([], 0),
    ([1], 1),
]
for values, expected in examples:
    root = build_tree(values)
    print(f"root={values} -> dựng lại {to_list(root)} -> kỳ vọng {expected}")

root=[1, 2, 3, 4, 5, 6] -> dựng lại [1, 2, 3, 4, 5, 6] -> kỳ vọng 6
root=[] -> dựng lại [] -> kỳ vọng 0
root=[1] -> dựng lại [1] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Cây nhị phân, tầng và chiều cao

**Cây nhị phân:** mỗi nút có tối đa hai con (`left`, `right`). **Tầng (level)** của nút là khoảng cách tới gốc (gốc ở tầng `0`). **Chiều cao tính theo số tầng** của cây: cây chỉ có gốc có `1` tầng. Ở notebook này ta đếm **số tầng** (gốc = tầng thứ `1`) để công thức cây hoàn hảo gọn: `h` tầng đầy ⇒ `2^h - 1` nút.

In [2]:
root = build_tree([1, 2, 3, 4, 5, 6, 7])           # cây hoàn hảo 3 tầng
print(root.val, root.left.val, root.right.val)     # 1 2 3
print(root.left.left.val, root.right.right.val)    # 4 7
print("None có con không?", root.left.left.left)   # None: nút lá không có con

1 2 3
4 7
None có con không? None


### 3.2. Ba loại cây: bất kỳ / hoàn chỉnh / hoàn hảo

- **Hoàn hảo (perfect):** mọi tầng đều đầy; `h` tầng có đúng `2^h - 1` nút.
- **Hoàn chỉnh (complete):** mọi tầng đầy **trừ tầng cuối**, tầng cuối **dồn trái**. Perfect ⊂ complete.
- **Bất kỳ:** không có ràng buộc.

Chỉ **cây hoàn chỉnh** mới cho phép thuật toán dưới `O(n)` của bài này.

In [3]:
perfect = build_tree([1, 2, 3, 4, 5, 6, 7])
complete = build_tree([1, 2, 3, 4, 5, 6])
not_complete = build_tree([1, 2, 3, 4, None, None, 5])     # nút 3 có con phải nhưng không có con trái


def is_complete(root):
    """Duyệt theo tầng; nếu đã thấy None thì phía sau không được còn nút."""
    queue, seen_none = deque([root]), False
    while queue:
        node = queue.popleft()
        if node is None:
            seen_none = True
        else:
            if seen_none:
                return False
            queue.append(node.left)
            queue.append(node.right)
    return True


print(is_complete(perfect), is_complete(complete), is_complete(not_complete))   # True True False
for h in range(1, 6):
    print(f"cây hoàn hảo {h} tầng: {2 ** h - 1} nút")

True True False
cây hoàn hảo 1 tầng: 1 nút
cây hoàn hảo 2 tầng: 3 nút
cây hoàn hảo 3 tầng: 7 nút
cây hoàn hảo 4 tầng: 15 nút
cây hoàn hảo 5 tầng: 31 nút


### 3.3. Đệ quy trên cây

Hàm đệ quy trên cây thường có dạng: **nếu `root` là `None` thì trả giá trị cơ sở**, ngược lại kết hợp kết quả của `root.left` và `root.right`. Số nút: `count(root) = 0` nếu rỗng, ngược lại `1 + count(left) + count(right)`.

In [4]:
def count_demo(root):
    if root is None:
        return 0
    return 1 + count_demo(root.left) + count_demo(root.right)


print(count_demo(build_tree([1, 2, 3, 4, 5, 6])))   # 6
print(count_demo(None))                              # 0

6
0


### 3.4. Chiều cao theo "gai trái" và "gai phải"

**Gai trái** là đường đi chỉ qua `left` từ gốc xuống; **gai phải** chỉ qua `right`. Số tầng theo gai trái cho biết **số tầng của cả cây hoàn chỉnh** (vì tầng cuối dồn trái, nút ngoài cùng bên trái sâu nhất). Với cây hoàn chỉnh, gai trái luôn **dài bằng hoặc hơn** gai phải, và nếu hai gai **bằng nhau** thì cây đó là **hoàn hảo**.

In [5]:
def left_levels(node):
    h = 0
    while node:
        h += 1
        node = node.left
    return h


def right_levels(node):
    h = 0
    while node:
        h += 1
        node = node.right
    return h


for values in ([1, 2, 3, 4, 5, 6, 7], [1, 2, 3, 4, 5, 6], [1, 2, 3, 4]):
    r = build_tree(values)
    print(values, "gai trái:", left_levels(r), "gai phải:", right_levels(r), "-> hoàn hảo?", left_levels(r) == right_levels(r))

[1, 2, 3, 4, 5, 6, 7] gai trái: 3 gai phải: 3 -> hoàn hảo? True
[1, 2, 3, 4, 5, 6] gai trái: 3 gai phải: 2 -> hoàn hảo? False
[1, 2, 3, 4] gai trái: 3 gai phải: 2 -> hoàn hảo? False


### 3.5. Dịch bit: `1 << h` là `2^h`

`1 << h` dịch bit `1` sang trái `h` vị trí, bằng `2^h`. Rất nhanh và là cách viết quen thuộc để tính lũy thừa của 2. Cây hoàn hảo `h` tầng có `(1 << h) - 1` nút.

In [6]:
for h in range(0, 6):
    print(h, 1 << h, 2 ** h, (1 << h) - 1)

0 1 1 0
1 2 2 1
2 4 4 3
3 8 8 7
4 16 16 15
5 32 32 31


### 3.6. Chặt nhị phân (binary search) trên một dãy "có rồi không"

Giả sử dãy `exists(0), exists(1), ..., exists(m)` có dạng `True, True, ..., True, False, ..., False` (đơn điệu). Muốn tìm **chỉ số `True` cuối cùng**, ta chặt nhị phân với `mid = (lo + hi + 1) // 2` (**làm tròn lên**) để tránh lặp vô hạn khi `lo = mid`.

In [7]:
flags = [True] * 6 + [False] * 4          # chỉ số True cuối cùng là 5
lo, hi = 0, len(flags) - 1
while lo < hi:
    mid = (lo + hi + 1) // 2              # làm tròn lên
    print(f"lo={lo}, hi={hi}, mid={mid}, exists(mid)={flags[mid]}")
    if flags[mid]:
        lo = mid
    else:
        hi = mid - 1
print("chỉ số True cuối cùng:", lo)

lo=0, hi=9, mid=5, exists(mid)=True
lo=5, hi=9, mid=7, exists(mid)=False
lo=5, hi=6, mid=6, exists(mid)=False
chỉ số True cuối cùng: 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét cây `[1,2,3,4,5,6]`:

```
        1
      /   \
     2     3
    / \   /
   4   5 6
```

- Tầng 1 có `1` nút, tầng 2 có `2` nút, tầng 3 (cuối) có `3` nút. Tổng `1 + 2 + 3 = 6`.
- Nếu tầng cuối **đầy** (`4` nút) thì cây hoàn hảo 3 tầng có `2^3 - 1 = 7` nút. Ta chỉ có `6`: **thiếu 1 nút** ở tận cùng bên phải của tầng cuối.

**Ý then chốt:** cây hoàn chỉnh = một cây hoàn hảo (các tầng trên) cộng **một phần tầng cuối dồn trái**. Số nút các tầng trên (`h - 1` tầng đầy) là **`2^(h-1) - 1`** — biết ngay không cần duyệt. Chỉ cần biết **tầng cuối có bao nhiêu nút**: một số trong `1 .. 2^(h-1)`, và các nút đó là dãy **liên tiếp từ trái**. Ta có thể **tìm** số đó bằng chặt nhị phân thay vì đếm từng cái.

Cách nghĩ thứ hai: **mọi cây con của cây hoàn chỉnh, hoặc là hoàn hảo, hoặc là hoàn chỉnh chưa đầy**. Nếu nhận ra cây con hoàn hảo bằng so sánh hai gai (rẻ: `O(h)`), ta đếm nó ngay bằng công thức `2^h - 1`, chỉ còn **một** nhánh phải đi tiếp.

## 5. Cách 1 — Brute Force / baseline: duyệt mọi nút (DFS)

**Ý tưởng.** Không dùng tính chất "hoàn chỉnh". Đếm mỗi nút một lần: `count(root) = 1 + count(left) + count(right)`.

In [8]:
def count_dfs(root):
    if root is None:
        return 0
    return 1 + count_dfs(root.left) + count_dfs(root.right)


for values, expected in examples:
    print(values, "->", count_dfs(build_tree(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5, 6] -> 6 (kỳ vọng 6 )
[] -> 0 (kỳ vọng 0 )
[1] -> 1 (kỳ vọng 1 )


In [9]:
# Biến thể lặp bằng stack (không đệ quy)
def count_dfs_iterative(root):
    count, stack = 0, [root] if root else []
    while stack:
        node = stack.pop()
        count += 1
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    return count


for values, expected in examples:
    print(values, "->", count_dfs_iterative(build_tree(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5, 6] -> 6 (kỳ vọng 6 )
[] -> 0 (kỳ vọng 0 )
[1] -> 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Gọi `n` là số nút. Mỗi nút được thăm đúng một lần: **Time `O(n)`**. **Space `O(h)`** cho ngăn xếp đệ quy với `h = O(log n)` ở cây hoàn chỉnh (vì cây rất cân bằng); với cây lệch bất kỳ có thể `O(n)`.

**Nút thắt.** Với `n = 5·10^4` cách này vẫn chạy nhanh, nhưng **đề yêu cầu dưới `O(n)`**: ta đang bỏ phí thông tin "cây hoàn chỉnh". Cần khai thác nó để **đếm mà không thăm từng nút**.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 (nhận ra cây con hoàn hảo):** với cây hoàn chỉnh `T`, so **số tầng theo gai trái `lh`** với **số tầng theo gai phải `rh`**:

- Nếu `lh == rh`: `T` là **hoàn hảo**, có đúng `2^lh - 1` nút — **đếm xong ngay** bằng công thức.
- Nếu `lh != rh` (thực ra `lh = rh + 1`): ta chưa biết, nhưng **một trong hai cây con** của gốc là hoàn hảo và cây con còn lại là hoàn chỉnh. Ta gọi đệ quy cho cả hai; nhánh hoàn hảo dừng ngay ở bước tiếp theo, nhánh kia đi tiếp xuống một tầng.

Mỗi tầng tốn `O(h)` để tính hai gai và chỉ **một** nhánh đi tiếp, `h` tầng ⇒ **`O(h²) = O(log² n)`**.

**Quan sát 2 (chặt nhị phân trên tầng cuối):** biết `h` (qua gai trái), các tầng trên có `2^(h-1) - 1` nút; tầng cuối có các vị trí `0 .. 2^(h-1) - 1` (từ trái sang phải) và nút có mặt là một **tiền tố liên tiếp**. Kiểm tra "vị trí `idx` có nút không" bằng cách đi từ gốc xuống theo các bit của `idx` (`O(h)`); chặt nhị phân tìm vị trí có mặt cuối cùng: `O(log n)` lần kiểm tra × `O(h)` ⇒ **`O(log² n)`**.

## 7. Các cách tiếp theo

### Cách 2 — BFS theo tầng (vẫn `O(n)`, đối chiếu)

**Giải thích.** Duyệt theo từng tầng bằng hàng đợi (`deque`), đếm số nút mỗi tầng. Vẫn thăm mọi nút nên `O(n)`; ta đưa vào để so sánh và làm đối chứng.

In [10]:
def count_bfs(root):
    if root is None:
        return 0
    count, queue = 0, deque([root])
    while queue:
        for _ in range(len(queue)):             # xử lý đúng một tầng
            node = queue.popleft()
            count += 1
            if node.left:
                queue.append(node.left)
            if node.right:
                queue.append(node.right)
    return count


for values, expected in examples:
    print(values, "->", count_bfs(build_tree(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5, 6] -> 6 (kỳ vọng 6 )
[] -> 0 (kỳ vọng 0 )
[1] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(w)`** với `w` là bề rộng lớn nhất (≈ `n/2` ở tầng cuối): **`O(n)`**.

### Cách 3 — Nhận ra cây con hoàn hảo bằng hai gai (đệ quy)

**Giải thích.** `lh` = số tầng theo gai trái, `rh` = số tầng theo gai phải của gốc. Nếu `lh == rh` thì cây hoàn hảo: trả `(1 << lh) - 1`. Ngược lại trả `1 + count(left) + count(right)`.
#### Code demo

In [11]:
def count_spines(root):
    if root is None:
        return 0
    lh = 0
    node = root
    while node:                      # gai trái
        lh += 1
        node = node.left
    rh = 0
    node = root
    while node:                      # gai phải
        rh += 1
        node = node.right
    if lh == rh:                     # cây hoàn hảo: 2^lh - 1 nút
        return (1 << lh) - 1
    return 1 + count_spines(root.left) + count_spines(root.right)


for values, expected in examples:
    print(values, "->", count_spines(build_tree(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5, 6] -> 6 (kỳ vọng 6 )
[] -> 0 (kỳ vọng 0 )
[1] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** mỗi lần gọi tốn `O(h)` cho hai gai; chuỗi gọi "đi tiếp" có độ sâu `h`, còn nhánh hoàn hảo thoát ngay nên tổng **Time `O(log² n)`** (`h ≈ log₂ n`); **Space `O(log n)`** cho ngăn xếp.

### Cách 4 — Chặt nhị phân trên tầng cuối

**Giải thích.**

1. Tính `h` = số tầng bằng gai trái. Nếu `h == 0` (cây rỗng) trả `0`.
2. Các tầng `1 .. h-1` đầy: `2^(h-1) - 1` nút.
3. Tầng cuối có tối đa `2^(h-1)` nút, đánh số `0 .. 2^(h-1) - 1`. Nút có mặt là các vị trí `0 .. k-1` (dồn trái). Dùng chặt nhị phân tìm vị trí lớn nhất có mặt, `exists(idx)` kiểm tra bằng cách đi từ gốc: ở mỗi tầng, chia đoạn `[lo, hi]` thành hai nửa; `idx` ở nửa trái thì đi `left`, ngược lại đi `right`.
4. Đáp án `= (2^(h-1) - 1) + (vị_trí_cuối + 1)`.
#### Code demo

In [12]:
def count_binary_search(root):
    if root is None:
        return 0
    h = 0
    node = root
    while node:
        h += 1
        node = node.left
    # tầng cuối (tầng thứ h) có vị trí 0 .. 2^(h-1) - 1
    last_level_size = 1 << (h - 1)

    def exists(idx):
        lo, hi = 0, last_level_size - 1
        node = root
        for _ in range(h - 1):                  # đi xuống h-1 bước để tới tầng cuối
            mid = (lo + hi) // 2
            if idx <= mid:
                node = node.left
                hi = mid
            else:
                node = node.right
                lo = mid + 1
        return node is not None

    lo, hi = 0, last_level_size - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2               # làm tròn lên
        if exists(mid):
            lo = mid
        else:
            hi = mid - 1
    return (last_level_size - 1) + (lo + 1)    # các tầng trên + số nút ở tầng cuối


for values, expected in examples:
    print(values, "->", count_binary_search(build_tree(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5, 6] -> 6 (kỳ vọng 6 )
[] -> 0 (kỳ vọng 0 )
[1] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** `exists` tốn `O(h)`; chặt nhị phân thực hiện `O(h)` lần (vì tầng cuối có tối đa `2^(h-1)` vị trí ⇒ `log₂ = h - 1` bước); **Time `O(h²) = O(log² n)`**, **Space `O(1)`**.

### Cách 5 — Lặp không đệ quy: so hai gai của hai cây con

**Giải thích.** Ở mỗi nút hiện tại, tính số tầng gai trái của **cây con trái** `lh` và của **cây con phải** `rh`:

- Nếu `lh == rh`: cây con **trái** hoàn hảo (có `2^lh - 1` nút), cộng gốc: `2^lh`; đi tiếp vào cây con **phải**.
- Ngược lại (`lh = rh + 1`): cây con **phải** hoàn hảo (`2^rh - 1` nút), cộng gốc: `2^rh`; đi tiếp vào cây con **trái**.

Vòng lặp kết thúc khi nút hiện tại là `None`.
#### Code demo

In [13]:
def count_iterative(root):
    def levels(node):                 # số tầng theo gai trái
        h = 0
        while node:
            h += 1
            node = node.left
        return h

    total = 0
    node = root
    while node:
        lh, rh = levels(node.left), levels(node.right)
        if lh == rh:
            total += 1 << lh          # gốc + cây con trái hoàn hảo (2^lh - 1)
            node = node.right
        else:
            total += 1 << rh          # gốc + cây con phải hoàn hảo (2^rh - 1)
            node = node.left
    return total


for values, expected in examples:
    print(values, "->", count_iterative(build_tree(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 5, 6] -> 6 (kỳ vọng 6 )
[] -> 0 (kỳ vọng 0 )
[1] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(log² n)`** (tối đa `h` vòng, mỗi vòng hai gai `O(h)`), **Space `O(1)`**.

> Cách 5 là bản nộp gọn, không đệ quy, `O(1)` bộ nhớ. Cách 3 dễ giải thích nhất; Cách 4 minh họa tư duy "chặt nhị phân trên tầng cuối".

## 8. TRACE từng bước

Ta trace Cách 3 (hai gai) và Cách 4 (chặt nhị phân) trên cây `[1,2,3,4,5,6]`.

In [14]:
def trace_spines(root, depth=0):
    pad = "  " * depth
    if root is None:
        print(f"{pad}count(None) = 0")
        return 0
    lh, node = 0, root
    while node:
        lh, node = lh + 1, node.left
    rh, node = 0, root
    while node:
        rh, node = rh + 1, node.right
    print(f"{pad}nút {root.val}: gai trái={lh}, gai phải={rh}", end="")
    if lh == rh:
        print(f" -> bằng nhau: cây hoàn hảo, {(1 << lh) - 1} nút")
        return (1 << lh) - 1
    print(" -> khác nhau: 1 + count(trái) + count(phải)")
    left = trace_spines(root.left, depth + 1)
    right = trace_spines(root.right, depth + 1)
    total = 1 + left + right
    print(f"{pad}=> nút {root.val}: 1 + {left} + {right} = {total}")
    return total


print("TỔNG:", trace_spines(build_tree([1, 2, 3, 4, 5, 6])))

nút 1: gai trái=3, gai phải=2 -> khác nhau: 1 + count(trái) + count(phải)
  nút 2: gai trái=2, gai phải=2 -> bằng nhau: cây hoàn hảo, 3 nút
  nút 3: gai trái=2, gai phải=1 -> khác nhau: 1 + count(trái) + count(phải)
    nút 6: gai trái=1, gai phải=1 -> bằng nhau: cây hoàn hảo, 1 nút
    count(None) = 0
  => nút 3: 1 + 1 + 0 = 2
=> nút 1: 1 + 3 + 2 = 6
TỔNG: 6


In [15]:
def trace_binary_search(root):
    h, node = 0, root
    while node:
        h, node = h + 1, node.left
    size = 1 << (h - 1)
    print(f"số tầng h={h}; các tầng trên có {size - 1} nút; tầng cuối có tối đa {size} vị trí (0..{size - 1})")

    def exists(idx):
        lo, hi, node = 0, size - 1, root
        path = []
        for _ in range(h - 1):
            mid = (lo + hi) // 2
            if idx <= mid:
                node, hi = node.left, mid
                path.append("L")
            else:
                node, lo = node.right, mid + 1
                path.append("R")
        return node is not None, "".join(path)

    lo, hi = 0, size - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok, path = exists(mid)
        print(f"  lo={lo}, hi={hi}, mid={mid}: đường đi {path} -> {'có nút' if ok else 'rỗng'}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    total = (size - 1) + (lo + 1)
    print(f"vị trí cuối có nút = {lo} -> tổng = {size - 1} + {lo + 1} = {total}")
    return total


trace_binary_search(build_tree([1, 2, 3, 4, 5, 6]))

số tầng h=3; các tầng trên có 3 nút; tầng cuối có tối đa 4 vị trí (0..3)
  lo=0, hi=3, mid=2: đường đi RL -> có nút
  lo=2, hi=3, mid=3: đường đi RR -> rỗng
vị trí cuối có nút = 2 -> tổng = 3 + 3 = 6


6

Bảng tóm tắt Cách 3 (cây `[1,2,3,4,5,6]`):

| nút | gai trái | gai phải | kết luận |
|---|---|---|---|
| 1 (gốc) | 3 | 2 | khác nhau → `1 + count(2) + count(3)` |
| 2 | 2 | 2 | hoàn hảo, `2^2 - 1 = 3` nút |
| 3 | 2 | 1 | khác nhau → `1 + count(6) + count(None)` |
| 6 | 1 | 1 | hoàn hảo, `2^1 - 1 = 1` nút |

Tổng: `1 + 3 + (1 + 1 + 0) = 6`. Chỉ nhìn các gai (không thăm hết từng nút) mà vẫn đếm đúng.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 5 (lặp, `O(log² n)` thời gian, `O(1)` bộ nhớ). Trên LeetCode `TreeNode` và `Optional` đã có sẵn; trong notebook ta đã tự định nghĩa ở trên.

In [16]:
class Solution:
    def countNodes(self, root: Optional[TreeNode]) -> int:
        def levels(node: Optional[TreeNode]) -> int:     # số tầng theo gai trái
            h = 0
            while node:
                h += 1
                node = node.left
            return h

        total = 0
        node = root
        while node:
            lh, rh = levels(node.left), levels(node.right)
            if lh == rh:                # cây con trái hoàn hảo: 2^lh - 1 nút, cộng gốc
                total += 1 << lh
                node = node.right
            else:                       # cây con phải hoàn hảo: 2^rh - 1 nút, cộng gốc
                total += 1 << rh
                node = node.left
        return total


sol = Solution()
for values, expected in examples:
    assert sol.countNodes(build_tree(values)) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Hai gai nhận ra cây hoàn hảo (Cách 3):** trong cây **hoàn chỉnh**, nếu số tầng theo gai trái bằng số tầng theo gai phải thì tầng cuối của nhánh phải cũng có nút, tức tầng cuối **đầy**, nên cây hoàn hảo: `2^lh - 1` nút. Ngược lại tầng cuối chưa đầy.
- **Một trong hai cây con luôn hoàn hảo (Cách 5):** gọi `lh`, `rh` là số tầng gai trái của cây con trái / phải. Vì tầng cuối dồn trái, `lh ∈ {rh, rh + 1}`.
  - Nếu `lh == rh`: tầng cuối đã chạm **hết cây con trái** (cây con trái đầy theo `lh` tầng) — cây con trái hoàn hảo; phần chưa biết nằm ở cây con phải.
  - Nếu `lh == rh + 1`: tầng cuối **không chạm tới** cây con phải, nên cây con phải chỉ có `rh` tầng và **đầy** — hoàn hảo; phần chưa biết nằm ở cây con trái.
  Cộng gốc `1` và `2^k - 1` nút của cây con hoàn hảo cho `2^k`.
- **Chặt nhị phân (Cách 4):** nút ở tầng cuối là tiền tố liên tiếp từ trái, nên `exists(idx)` có dạng `True...True False...False`; đi theo đoạn `[lo, hi]` chia đôi chính là đọc các bit của `idx` từ cao xuống thấp — đúng đường từ gốc tới vị trí `idx`.
- **Kết thúc:** mỗi bước đi xuống một tầng (Cách 3, 5) hoặc thu hẹp đoạn (Cách 4); cây hữu hạn nên dừng.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h = O(log n)` là số tầng.

| Cách | Time | Space |
|---|---|---|
| 1. DFS | `O(n)` | `O(h)` (ngăn xếp) |
| 2. BFS | `O(n)` | `O(n)` (hàng đợi) |
| 3. Hai gai, đệ quy | `O(log² n)` | `O(log n)` |
| 4. Chặt nhị phân tầng cuối | `O(log² n)` | `O(1)` |
| 5. Lặp hai gai | `O(log² n)` | `O(1)` |

**Vì sao `log² n`:** một "lượt" tính hai gai tốn `O(h)`; số lượt tối đa `h` (mỗi lượt đi xuống một tầng) ⇒ `O(h²)`. Với `n = 5·10^4`: `h ≈ 16`, `h² ≈ 256` phép — so với `5·10^4` phép của DFS.

Không có khác biệt "average/worst" lớn: worst case là khi mọi lượt đều phải so gai (cây hoàn chỉnh chưa đầy ở mọi mức); kết quả luôn nằm trong `O(log² n)`.

## 12. Bộ test

Vì ta **tạo** cây hoàn chỉnh theo số nút `n`, ta biết đáp án đúng chính là `n`. Thêm một `oracle` độc lập: duyệt cây bằng **stack tự quản** rồi đếm (không dùng bất kỳ tính chất "hoàn chỉnh" nào). Ta đối chiếu mọi cách với cả hai.

In [17]:
def build_complete_tree(n):
    """Cây hoàn chỉnh có đúng n nút: nút i có con 2i+1, 2i+2 (nếu < n)."""
    if n == 0:
        return None
    nodes = [TreeNode(i + 1) for i in range(n)]
    for i in range(n):
        if 2 * i + 1 < n:
            nodes[i].left = nodes[2 * i + 1]
        if 2 * i + 2 < n:
            nodes[i].right = nodes[2 * i + 2]
    return nodes[0]


def oracle(root):
    """Đếm bằng stack tự quản, độc lập với mọi tính chất của cây hoàn chỉnh."""
    seen_values_count = 0
    stack = []
    if root is not None:
        stack.append(root)
    while stack:
        node = stack.pop()
        seen_values_count += 1
        for child in (node.right, node.left):
            if child is not None:
                stack.append(child)
    return seen_values_count


impls = {
    "dfs": count_dfs,
    "dfs_iterative": count_dfs_iterative,
    "bfs": count_bfs,
    "spines": count_spines,
    "binary_search": count_binary_search,
    "iterative": count_iterative,
    "submit": sol.countNodes,
}

for values, expected in examples:
    root = build_tree(values)
    assert oracle(root) == expected and is_complete(root)
    for key, fn in impls.items():
        assert fn(root) == expected, (key, values, fn(root), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [18]:
edge_cases = [
    ([], 0, "cây rỗng"),
    ([1], 1, "một nút"),
    ([1, 2], 2, "gốc và con trái"),
    ([1, 2, 3], 3, "cây hoàn hảo 2 tầng"),
    ([1, 2, 3, 4], 4, "tầng cuối chỉ có 1 nút"),
    ([1, 2, 3, 4, 5], 5, "tầng cuối 2 nút"),
    ([1, 2, 3, 4, 5, 6, 7], 7, "cây hoàn hảo 3 tầng"),
    (list(range(1, 16)), 15, "cây hoàn hảo 4 tầng"),
    (list(range(1, 17)), 16, "vừa qua cây hoàn hảo: tầng cuối 1 nút"),
    (list(range(1, 32)), 31, "cây hoàn hảo 5 tầng"),
]
for values, expected, label in edge_cases:
    root = build_tree(values)
    assert oracle(root) == expected and is_complete(root), label
    for key, fn in impls.items():
        assert fn(root) == expected, (key, label, fn(root), expected)
    print("PASS", label)

PASS cây rỗng
PASS một nút
PASS gốc và con trái
PASS cây hoàn hảo 2 tầng
PASS tầng cuối chỉ có 1 nút
PASS tầng cuối 2 nút
PASS cây hoàn hảo 3 tầng
PASS cây hoàn hảo 4 tầng
PASS vừa qua cây hoàn hảo: tầng cuối 1 nút
PASS cây hoàn hảo 5 tầng


### 12.2. Tính chất và phản ví dụ

- Với **mọi** `n` từ `0` đến vài trăm, cây hoàn chỉnh `n` nút phải cho đáp án `n`.
- Cây có `2^h - 1` nút là hoàn hảo: hai gai bằng nhau; thêm đúng một nút thì hai gai khác nhau.
- **Phản ví dụ:** các thuật toán `O(log² n)` **chỉ đúng với cây hoàn chỉnh**. Với cây không hoàn chỉnh `[1,2,3,4,None,None,5]` (nút `3` chỉ có con phải), Cách 3 có thể trả sai (đúng là `5` nút).

In [19]:
for n in range(0, 301):
    root = build_complete_tree(n)
    assert is_complete(root) and oracle(root) == n
    for key, fn in impls.items():
        assert fn(root) == n, (key, n, fn(root))
print("PASS n = 0..300: mọi cách cho đúng n")

for h in range(1, 9):
    perfect_root = build_complete_tree((1 << h) - 1)
    assert left_levels(perfect_root) == right_levels(perfect_root) == h
    bigger = build_complete_tree(1 << h)
    assert left_levels(bigger) != right_levels(bigger)
print("PASS tính chất: 2^h - 1 nút => hai gai bằng nhau; thêm một nút => khác nhau")

nc = build_tree([1, 2, 3, 4, None, None, 5])
print("cây không hoàn chỉnh:", to_list(nc), "| is_complete =", is_complete(nc))
print("DFS (đúng):", count_dfs(nc), "| hai gai (Cách 3):", count_spines(nc), "| lặp (Cách 5):", count_iterative(nc))
assert count_dfs(nc) == 5 and count_spines(nc) != 5

PASS n = 0..300: mọi cách cho đúng n
PASS tính chất: 2^h - 1 nút => hai gai bằng nhau; thêm một nút => khác nhau
cây không hoàn chỉnh: [1, 2, 3, 4, None, None, 5] | is_complete = False
DFS (đúng): 5 | hai gai (Cách 3): 7 | lặp (Cách 5): 4


### 12.3. Đối chiếu ngẫu nhiên (có seed) và cây lớn

In [20]:
random.seed(222)
for trial in range(400):
    n = random.randint(0, 5000)
    root = build_complete_tree(n)
    for key, fn in impls.items():
        assert fn(root) == n, (key, n, fn(root))
print("PASS 400 cây hoàn chỉnh ngẫu nhiên (n tới 5000): mọi cách khớp n")

# Cây lớn nhất của đề (n = 5 * 10^4) và các cận đáng chú ý
for n in (49999, 50000, 32767, 32768, 32769, 16383, 16384):
    root = build_complete_tree(n)
    for key, fn in impls.items():
        assert fn(root) == n, (key, n, fn(root))
print("PASS n gần 2^k - 1 / 2^k và n = 5*10^4")

# Dùng build_tree (định dạng LeetCode) để dựng cùng cây rồi so sánh
random.seed(2220)
for trial in range(100):
    n = random.randint(1, 300)
    values = list(range(1, n + 1))                  # level-order của cây hoàn chỉnh không có None
    root = build_tree(values)
    assert is_complete(root) and to_list(root) == values
    assert sol.countNodes(root) == n
print("PASS 100 cây dựng từ danh sách level-order (build_tree) cho đúng n")

PASS 400 cây hoàn chỉnh ngẫu nhiên (n tới 5000): mọi cách khớp n
PASS n gần 2^k - 1 / 2^k và n = 5*10^4
PASS 100 cây dựng từ danh sách level-order (build_tree) cho đúng n


### 12.4. Đếm số bước để thấy `O(n)` so với `O(log² n)`

Ta đếm số lần **di chuyển qua một nút** (con trỏ `.left`/`.right`) của Cách 1 và Cách 5 để thấy chênh lệch (độc lập với tốc độ máy).

In [21]:
def steps_dfs(n):
    return n                                   # mỗi nút đúng một lần


def steps_iterative(root):
    steps = 0

    def levels(node):
        nonlocal steps
        h = 0
        while node:
            h += 1
            steps += 1
            node = node.left
        return h

    node = root
    while node:
        lh, rh = levels(node.left), levels(node.right)
        steps += 1
        node = node.right if lh == rh else node.left
    return steps


print(f"{'n':>8} {'DFS':>8} {'Cách 5':>8}")
for n in (10, 100, 1000, 10_000, 50_000):
    print(f"{n:>8} {steps_dfs(n):>8} {steps_iterative(build_complete_tree(n)):>8}")

       n      DFS   Cách 5
      10       10       14
     100      100       45
    1000     1000       96
   10000    10000      187
   50000    50000      246


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng công thức hoàn hảo mà không kiểm tra "hoàn chỉnh"
Xem phần 12.2: `count_spines` trả sai với cây không hoàn chỉnh. Đề **đảm bảo** cây hoàn chỉnh, nhưng khi áp dụng bài khác (cây bất kỳ) phải quay về DFS `O(n)`.

### Lỗi 2: Nhầm `2^h` với `2^h - 1`
#### Code sai

In [22]:
def sai_thieu_tru_1(root):
    if root is None:
        return 0
    lh = rh = 0
    node = root
    while node:
        lh, node = lh + 1, node.left
    node = root
    while node:
        rh, node = rh + 1, node.right
    if lh == rh:
        return 1 << lh                            # SAI: cây hoàn hảo lh tầng có 2^lh - 1 nút
    return 1 + sai_thieu_tru_1(root.left) + sai_thieu_tru_1(root.right)


root = build_tree([1, 2, 3, 4, 5, 6, 7])
print("đúng:", Solution().countNodes(root), "| sai:", sai_thieu_tru_1(root))

đúng: 7 | sai: 8


**Vì sao sai:** cây hoàn hảo `h` tầng có `2^h - 1` nút (cộng hết mọi tầng `1 + 2 + 4 + ... + 2^(h-1)`), không phải `2^h`. **Code đúng:** `(1 << lh) - 1`.

### Lỗi 3: Tính cả hai chiều cao bằng cùng một gai (sao chép nhầm)
#### Code sai

In [23]:
def sai_hai_gai_trai(root):
    if root is None:
        return 0
    lh = rh = 0
    node = root
    while node:
        lh, node = lh + 1, node.left
    node = root
    while node:
        rh, node = rh + 1, node.left              # SAI: gai phải phải đi theo node.right
    if lh == rh:                                  # luôn đúng => luôn trả (1 << lh) - 1
        return (1 << lh) - 1
    return 1 + sai_hai_gai_trai(root.left) + sai_hai_gai_trai(root.right)


root = build_tree([1, 2, 3, 4, 5, 6])
print("đúng:", Solution().countNodes(root), "| sai:", sai_hai_gai_trai(root))

đúng: 6 | sai: 7


**Vì sao sai:** nếu cả hai "chiều cao" đều đi theo `left` thì luôn bằng nhau, thuật toán lúc nào cũng cho rằng cây **hoàn hảo** và trả `2^h - 1` (ở đây `7` thay vì `6`). Lỗi sao chép–dán rất hay gặp; hãy tự trace trên một cây **chưa đầy** như `[1,2,3,4,5,6]`. Ngoài ra nhớ chọn **một quy ước** đếm (số tầng, cây rỗng `= 0`) và giữ xuyên suốt.

### Lỗi 4: Chặt nhị phân không làm tròn lên (lặp vô hạn)
#### Code sai

In [24]:
def sai_chat_nhi_phan(flags):
    """Tìm chỉ số True cuối cùng, nhưng mid làm tròn xuống: bị kẹt khi lo = mid."""
    lo, hi, guard = 0, len(flags) - 1, 0
    while lo < hi and guard < 20:                 # guard chỉ để demo, tránh treo notebook
        guard += 1
        mid = (lo + hi) // 2                      # SAI: khi lo + 1 == hi, mid == lo, gán lo = mid không đổi
        if flags[mid]:
            lo = mid
        else:
            hi = mid - 1
    return lo, guard


print(sai_chat_nhi_phan([True, True, False]))     # kẹt tới khi guard hết (guard = 20)

(1, 20)


**Vì sao sai:** khi dùng `lo = mid`, phải có `mid = (lo + hi + 1) // 2` (làm tròn lên) để `mid > lo`. **Code đúng:** xem Cách 4.

### Lỗi 5: Quên trường hợp cây rỗng
#### Code sai

In [25]:
def sai_quen_rong(root):
    h, node = 0, root
    while node:
        h, node = h + 1, node.left
    last_level_size = 1 << (h - 1)                # h = 0 -> 1 << -1: lỗi
    return last_level_size


try:
    sai_quen_rong(None)
except ValueError as e:
    print("ValueError:", e)

ValueError: negative shift count


**Vì sao sai:** với `root = None` thì `h = 0`, `1 << (h - 1)` là dịch bit âm → lỗi. **Code đúng:** `if root is None: return 0` ngay từ đầu.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**cây hoàn chỉnh / complete**" và yêu cầu "**dưới `O(n)`**" → nghĩ ngay **`O(log² n)`**: hai gai hoặc chặt nhị phân trên tầng cuối.
- Công thức cần nhớ: **cây hoàn hảo `h` tầng có `2^h - 1` nút**; số nút `n` của cây hoàn chỉnh `h` tầng nằm trong `[2^(h-1), 2^h - 1]`.
- Hai gai bằng nhau ⇒ **hoàn hảo**; khác nhau ⇒ một cây con hoàn hảo, đi tiếp vào cây con kia.
- Cây đã được đánh số theo kiểu **heap** (`i` có con `2i+1`, `2i+2`; hay `2i`, `2i+1` nếu đánh số từ 1): đường từ gốc tới vị trí `idx` là các **bit của `idx`** — mẹo dùng trong Cách 4.
- Cây **bất kỳ** → DFS/BFS `O(n)`; đừng dùng công thức hoàn hảo.
- Khi tự dựng cây để test: viết `build_tree` (level-order, `None` = rỗng), `to_list`, và hàm **sinh cây theo số nút `n`** để biết sẵn đáp án.

## 15. Biến thể của bài

1. **Cây bất kỳ**: đếm số nút `O(n)` (Cách 1); hoặc đếm nút lá, nút có đúng một con.
2. **Kiểm tra cây có hoàn chỉnh không** (LeetCode 958): BFS, sau `None` không được có nút.
3. **Chiều cao tối đa** của cây (104), **độ sâu nhỏ nhất** (111).
4. **Tìm nút thứ `k`** của cây hoàn chỉnh theo thứ tự level-order: đường đi theo bit của `k`.
5. **Cây hoàn hảo đếm nút** `2^h - 1`; **đếm nút ở tầng cuối** của cây hoàn chỉnh.
6. Cây đánh số theo **heap**: `n` nút ⇒ chiều cao `floor(log2 n) + 1` tầng.

In [26]:
def last_level_count(root):
    """Số nút ở tầng cuối của cây hoàn chỉnh (dùng đáp án Cách 5 trừ đi các tầng đầy phía trên)."""
    n = count_iterative(root)
    if n == 0:
        return 0
    h = left_levels(root)
    return n - ((1 << (h - 1)) - 1)


def kth_node_path(k):
    """Đường đi (L/R) từ gốc tới nút thứ k (k tính từ 1, theo level-order) trong cây hoàn chỉnh."""
    bits = bin(k)[3:]                       # bỏ '0b1': bit đầu là gốc
    return "".join("L" if b == "0" else "R" for b in bits)


print([last_level_count(build_complete_tree(n)) for n in (1, 2, 3, 4, 5, 6, 7, 8)])   # [1, 1, 2, 1, 2, 3, 4, 1]
print(kth_node_path(6), kth_node_path(5), kth_node_path(1))                           # RL LR (rỗng)

[1, 1, 2, 1, 2, 3, 4, 1]
RL LR 


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS | `1 + count(trái) + count(phải)` | `O(n)` | `O(h)` | cây bất kỳ; làm oracle |
| 2. BFS | duyệt theo tầng bằng hàng đợi | `O(n)` | `O(n)` | cần số nút mỗi tầng |
| 3. Hai gai, đệ quy | gai trái = gai phải ⇒ hoàn hảo `2^h - 1` | `O(log² n)` | `O(log n)` | dễ giải thích nhất |
| 4. Chặt nhị phân tầng cuối | tìm vị trí cuối có nút bằng `exists(idx)` | `O(log² n)` | `O(1)` | cần tư duy "đường đi theo bit" |
| 5. Lặp hai gai | luôn có một cây con hoàn hảo, cộng `2^k` rồi đi tiếp | `O(log² n)` | `O(1)` | **bản nộp gọn, không đệ quy** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `1 << 4`; (b) số nút của cây hoàn hảo `5` tầng; (c) `left_levels` và `right_levels` của cây `[1,2,3,4,5,6]`; (d) `is_complete(build_tree([1,2,3,None,4]))`.

**Bài 2 (tính bằng tay).** Cho cây hoàn chỉnh `[1,2,3,4,5,6,7,8,9,10]` (10 nút). Tính số tầng `h`, số nút các tầng trên, và số nút ở tầng cuối; xác nhận tổng bằng `10`.

**Bài 3 (điền TODO).** Hoàn thiện `count_todo(root)` theo Cách 3 (hai gai, đệ quy).

In [27]:
def count_todo(root):
    # TODO 1: nếu root là None thì trả 0
    # TODO 2: tính lh = số tầng theo gai trái, rh = số tầng theo gai phải (vòng while)
    # TODO 3: nếu lh == rh thì trả (1 << lh) - 1
    # TODO 4: ngược lại trả 1 + count_todo(root.left) + count_todo(root.right)
    pass


# Tự làm xong rồi mới mở test:
# for values, expected in examples:
#     assert count_todo(build_tree(values)) == expected

**Bài 4 (tự viết lại).** Viết `count_bs2(root)` bằng **chặt nhị phân trên tầng cuối**, **không nhìn lại** Cách 4; kiểm tra với `build_complete_tree(n)` cho `n = 0..200`.

**Bài 5 (biến thể / mini challenge).** Viết `is_perfect(root)` kiểm tra cây có hoàn hảo không và `count_any_tree(root)` đếm nút cho **cây bất kỳ** (không giả định hoàn chỉnh). Sau đó giải thích vì sao với `[1,2,3,4,None,None,5]` kết quả của `count_spines` sai nhưng `count_any_tree` đúng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1 << h` là `2^h`; cây hoàn hảo `h` tầng có `2^h - 1` nút; gai trái đi theo `left`. Bài 2: tầng cuối còn lại sau khi trừ các tầng đầy phía trên. Bài 5: DFS không cần giả định gì.
- **Mức 2 (vừa):** Bài 1: `16`, `31`, `3` và `2`, `False` (nút `2` có con phải nhưng không có con trái). Bài 2: `n = 10` nằm trong `[8, 15]` ⇒ `h = 4` tầng; các tầng trên có `2^3 - 1 = 7` nút; tầng cuối có `10 - 7 = 3` nút. Bài 3: đúng khung Cách 3.
- **Mức 3 (sâu):** Bài 4: dùng `exists(idx)` đi `h - 1` bước và `mid = (lo + hi + 1) // 2`. Bài 5: `is_perfect`: `left_levels == right_levels` chỉ đủ cho cây hoàn chỉnh; với cây bất kỳ phải kiểm tra **mọi** cây con (hai cây con của mỗi nút phải hoàn hảo và có cùng số tầng). `count_spines` sai trên `[1,2,3,4,None,None,5]` vì giả định "tầng cuối dồn trái" bị vi phạm (nút `3` có con phải mà không có con trái), nên hai gai bằng nhau không còn suy ra cây hoàn hảo.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 1 << 4 == 16 and (1 << 5) - 1 == 31
r6 = build_tree([1, 2, 3, 4, 5, 6])
assert (left_levels(r6), right_levels(r6)) == (3, 2)
assert is_complete(build_tree([1, 2, 3, None, 4])) is False

# Bài 2
n = 10
h = left_levels(build_complete_tree(n))
assert h == 4
upper = (1 << (h - 1)) - 1
assert upper == 7 and n - upper == 3 and upper + (n - upper) == n


# Bài 3
def count_todo_ans(root):
    if root is None:
        return 0
    lh, node = 0, root
    while node:
        lh, node = lh + 1, node.left
    rh, node = 0, root
    while node:
        rh, node = rh + 1, node.right
    if lh == rh:
        return (1 << lh) - 1
    return 1 + count_todo_ans(root.left) + count_todo_ans(root.right)


for values, expected in examples:
    assert count_todo_ans(build_tree(values)) == expected


# Bài 4
def count_bs2(root):
    if root is None:
        return 0
    h, node = 0, root
    while node:
        h, node = h + 1, node.left
    size = 1 << (h - 1)

    def exists(idx):
        lo, hi, node = 0, size - 1, root
        for _ in range(h - 1):
            mid = (lo + hi) // 2
            if idx <= mid:
                node, hi = node.left, mid
            else:
                node, lo = node.right, mid + 1
        return node is not None

    lo, hi = 0, size - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if exists(mid):
            lo = mid
        else:
            hi = mid - 1
    return size - 1 + lo + 1


for n in range(0, 201):
    assert count_bs2(build_complete_tree(n)) == n


# Bài 5
def is_perfect(root):
    def height(node):                          # số tầng nếu cây con hoàn hảo, ngược lại -1
        if node is None:
            return 0
        hl, hr = height(node.left), height(node.right)
        if hl == -1 or hr == -1 or hl != hr:
            return -1
        return hl + 1

    return height(root) != -1


def count_any_tree(root):
    return 0 if root is None else 1 + count_any_tree(root.left) + count_any_tree(root.right)


assert is_perfect(build_tree([1, 2, 3, 4, 5, 6, 7])) and not is_perfect(build_tree([1, 2, 3, 4, 5, 6]))
assert count_any_tree(nc) == 5 != count_spines(nc)
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (lặp, so hai gai của hai cây con)

```python
class Solution:
    def countNodes(self, root: Optional[TreeNode]) -> int:
        def levels(node):                       # số tầng theo gai trái
            h = 0
            while node:
                h += 1
                node = node.left
            return h

        total, node = 0, root
        while node:
            lh, rh = levels(node.left), levels(node.right)
            if lh == rh:
                total += 1 << lh                # trái hoàn hảo + gốc
                node = node.right
            else:
                total += 1 << rh                # phải hoàn hảo + gốc
                node = node.left
        return total
```

**Checklist**

- [ ] Nhớ **cây hoàn hảo `h` tầng có `2^h - 1` nút** và chọn một quy ước đếm tầng (cây rỗng `= 0`)?
- [ ] Hai gai bằng nhau ⇒ hoàn hảo; khác nhau ⇒ đi tiếp vào cây con còn lại?
- [ ] Biết thuật toán `O(log² n)` **chỉ đúng với cây hoàn chỉnh** (đề đảm bảo), cây khác dùng DFS?
- [ ] Chặt nhị phân: `mid = (lo + hi + 1) // 2` khi gán `lo = mid`?
- [ ] Xử lý `root is None` ở đầu (hoặc vòng `while node` tự dừng)?
- [ ] Đã thử `[]`, `[1]`, cây hoàn hảo `2^h - 1` nút, `2^h` nút, và `n = 5·10^4`?
- [ ] Đã đối chiếu với duyệt cây `O(n)` và với số nút `n` biết sẵn khi sinh cây?
- [ ] Skeleton đúng `class Solution` / `def countNodes(self, root: Optional[TreeNode]) -> int`?